# SDG7 — Adım 3 / Güncel sürüm: London Smart Meters — Görülmemiş evlerde tahmin

**Neyi çözüyor?** Kaggle `hhblock_dataset.zip` için 404 hatası veren indirmeyi kullanmıyoruz. Bunun yerine orijinal Low Carbon London verilerinin *üçüncü taraf aynasındaki* yaklaşık 50,8 MB'lık tek bir CSV bölümünü indiriyoruz. Tam 5.567 hanenin tamamını kullandığımız anlamına gelmez.

Kaynak: [London Datastore / Greater London Authority](https://data.london.gov.uk/dataset/smartmeter-energy-consumption-data-in-london-households-vqm0d) ve [Hugging Face kaynak dosyası](https://huggingface.co/datasets/Sufiyan83/Low-Carbon-London-Smart-Meter-Cleaned-FeatureReady/blob/main/Partitioned%20LCL%20Data/Small%20LCL%20Data/LCL-June2015v2_0.csv).

**Çalıştırma:** Önce 1. hücreyi, sonra sırayla devam et. Kaggle girişi veya el ile dosya yükleme gerekmez. İndirme başarısızsa hücredeki hata metnini ilet; aynı hücreyi tekrar tekrar başlatma.

**Amaç:** Modeli belli hanelerde eğitip eğitimde bulunmayan hanelerin daha ileri tarihli ölçümlerinde MAE/RMSE ile test etmek. Bu gerçek çok haneli bir örneklem denemesidir; İstanbul/Türkiye'den ölçüm değildir.


## 1 — Küçük gerçek London verisini indir (Kaggle 404 yolunu kullanmıyor)

In [ ]:
%pip -q install -U huggingface_hub
from pathlib import Path
from huggingface_hub import hf_hub_download
import pandas as pd

REPO = "Sufiyan83/Low-Carbon-London-Smart-Meter-Cleaned-FeatureReady"
FILE = "Partitioned LCL Data/Small LCL Data/LCL-June2015v2_0.csv"
source_file = Path(hf_hub_download(repo_id=REPO, filename=FILE, repo_type="dataset"))
print("✅ CSV indirildi:", source_file.name)
print("Dosya boyutu (MB):", round(source_file.stat().st_size/1e6,1))
print("Sütunlar:", pd.read_csv(source_file, nrows=3).columns.tolist())


## 2 — Dosyayı doğrula

In [ ]:
# Hızlı doğrulama: veri gerçek mi, kaç ev içeriyor?
import pandas as pd
preview = pd.read_csv(source_file, usecols=lambda x: x.strip().lower() in {"lclid", "datetime"}, low_memory=False)
print("İndirilen satır sayısı:", f"{len(preview):,}")
print("Farklı hane sayısı:", preview.iloc[:,0].nunique())
print("İlk 5 hane:", preview.iloc[:,0].dropna().unique()[:5])
del preview


## 3 — Eğitime girmeyen haneleri değerlendiren Random Forest

In [ ]:
"""Step 3: London Smart Meters — strict held-out households and a temporal cutoff.
Run: python london_unseen_home.py /path/to/block_0.csv
Results: London_Sonuclar/
"""
from __future__ import annotations
import argparse
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
MAX_HOMES = 12
MIN_HOURS = 24 * 75


def process(path: str | Path, out_dir: str | Path = 'London_Sonuclar'):
    output = Path(out_dir); output.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(path, low_memory=False)
    original_cols = list(df.columns)
    df.columns = [c.strip() for c in df.columns]
    lower = {c.lower(): c for c in df.columns}
    if 'lclid' not in lower or 'datetime' not in lower:
        raise ValueError(f'Beklenen LCLid ve DateTime sütunları bulunamadı. Gerçek sütunlar: {original_cols}')
    id_col, time_col = lower['lclid'], lower['datetime']
    kwh_cols = [c for c in df.columns if 'kwh' in c.lower()]
    if not kwh_cols:
        raise ValueError(f'kWh sütunu bulunamadı: {original_cols}')
    kwh_col = kwh_cols[0]
    dt = pd.to_datetime(df[time_col].astype(str).str[:19], errors='coerce')
    val = pd.to_numeric(df[kwh_col].replace('Null', np.nan), errors='coerce')
    base = pd.DataFrame({'home':df[id_col].astype(str), 'time':dt, 'half_kwh':val})
    base = base.dropna(subset=['home','time','half_kwh'])
    base = base[(base.half_kwh>=0) & (base.half_kwh <= 30)]
    base['hour'] = (base.time - pd.Timedelta(minutes=30)).dt.floor('h')
    # Only accept an hourly sum when two distinct half-hour timestamps are present.
    hourly = (base.groupby(['home','hour'],as_index=False)
            .agg(kwh=('half_kwh','sum'),slots=('time','nunique')))
    hourly = hourly[hourly.slots == 2].drop(columns='slots')
    counts = hourly.groupby('home').size().sort_values(ascending=False)
    homes = counts[counts >= MIN_HOURS].head(MAX_HOMES).index.tolist()
    if len(homes) < 4:
        raise ValueError(f'En az 4 uygun hane gerekli; {len(homes)} bulundu. Daha fazla haneyi içeren bir block_*.csv yükleyin. İlk hane sayıları: {counts.head().to_dict()}')
    hourly = hourly[hourly.home.isin(homes)].copy()
    # Reindex to guarantee that a lag of 24 truly means the same hour yesterday.
    parts = []
    for home, group in hourly.groupby('home'):
        s = group.set_index('hour').kwh.sort_index()
        s = s[~s.index.duplicated(keep='first')]
        s = s.reindex(pd.date_range(s.index.min(),s.index.max(),freq='h'))
        frame = pd.DataFrame({'home':home,'hour':s.index,'kwh':s.to_numpy()})
        parts.append(frame)
    data = pd.concat(parts, ignore_index=True).sort_values(['home','hour'])
    g = data.groupby('home',sort=False)['kwh']
    for lag in [24,48,168]: data[f'lag_{lag}'] = g.shift(lag)
    # Only values from the previous day and earlier are included: no forecast-day readings.
    data['past_24h_mean'] = data.groupby('home',sort=False)['kwh'].transform(lambda s: s.shift(24).rolling(24,min_periods=24).mean())
    data['past_7d_mean'] = data.groupby('home',sort=False)['kwh'].transform(lambda s: s.shift(24).rolling(168,min_periods=168).mean())
    data['tod'] = data.hour.dt.hour
    data['weekday'] = data.hour.dt.dayofweek
    data['month'] = data.hour.dt.month
    data['weekend'] = (data.weekday>=5).astype(int)
    data['hour_sin'] = np.sin(2*np.pi*data.tod/24)
    data['hour_cos'] = np.cos(2*np.pi*data.tod/24)
    feature_cols = ['lag_24','lag_48','lag_168','past_24h_mean','past_7d_mean','tod','weekday','month','weekend','hour_sin','hour_cos']
    # All feature lags must be available, and all labels have measured consumption.
    clean = data.dropna(subset=feature_cols+['kwh']).copy()
    train_homes, heldout_homes = [], []
    r = np.random.default_rng(SEED)
    shuffled = r.permutation(homes)
    n_heldout = max(1,int(np.ceil(len(homes)*0.25)))
    heldout_homes = sorted(shuffled[:n_heldout].tolist())
    train_homes = sorted(shuffled[n_heldout:].tolist())
    dates = pd.DatetimeIndex(clean.hour.dt.floor('D').unique()).sort_values()
    if len(dates) < 30: raise ValueError(f'Yeterli farklı gün yok: {len(dates)}')
    cutoff = dates[int(len(dates)*0.8)]
    train = clean[(clean.home.isin(train_homes)) & (clean.hour < cutoff)]
    test = clean[(clean.home.isin(heldout_homes)) & (clean.hour >= cutoff)]
    if len(train)<200 or len(test)<100:
        raise ValueError(f'Yeterli eğitim/test gözlemi yok: eğitim={len(train)}, test={len(test)}. Daha uzun dönem içeren dosya seçin.')
    # Numeric lag/cycle features have no home identifier: new home prediction possible.
    rf = RandomForestRegressor(n_estimators=80,max_depth=18,min_samples_leaf=3,
                               random_state=SEED,n_jobs=-1)
    rf.fit(train[feature_cols],train.kwh)
    test = test.copy()
    test['prediction_kwh'] = rf.predict(test[feature_cols]).clip(min=0)
    test['baseline_yesterday'] = test.lag_24
    test['baseline_lastweek'] = test.lag_168
    rows=[]
    for m, col in [('Random Forest','prediction_kwh'),('Previous day','baseline_yesterday'),('Previous week','baseline_lastweek')]:
        rows.append({'method':m,'mae_kwh':mean_absolute_error(test.kwh,test[col]),
                     'rmse_kwh':np.sqrt(mean_squared_error(test.kwh,test[col]))})
    metrics = pd.DataFrame(rows)
    per_home=(test.groupby('home').apply(lambda x:pd.Series({
        'rows':len(x),'mae_kwh':mean_absolute_error(x.kwh,x.prediction_kwh),
        'rmse_kwh':np.sqrt(mean_squared_error(x.kwh,x.prediction_kwh))
    }),include_groups=False).reset_index())
    metrics.to_csv(output/'metrics_unseen_homes.csv',index=False)
    per_home.to_csv(output/'per_home_metrics.csv',index=False)
    test[['home','hour','kwh','prediction_kwh','baseline_yesterday','baseline_lastweek']].to_csv(output/'unseen_home_predictions.csv',index=False)
    samplehome = heldout_homes[0]
    sample = test[test.home == samplehome].copy()
    sample['date'] = sample.hour.dt.date
    target_date = sample.groupby('date').size().sort_values(ascending=False).index[0]
    sample = sample[sample.date == target_date].sort_values('hour')
    fig,ax=plt.subplots(figsize=(10,4))
    ax.plot(sample.hour.dt.hour,sample.kwh,label='Actual',marker='o',markersize=3)
    ax.plot(sample.hour.dt.hour,sample.prediction_kwh,label='RF prediction',marker='o',markersize=3)
    ax.set(title=f'Unseen home {samplehome} | {target_date}',xlabel='Hour',ylabel='kWh (hour)')
    ax.legend(); fig.tight_layout(); fig.savefig(output/'unseen_home_example.png',dpi=170);plt.close(fig)
    note = (
        'London Smart Meters — held-out homes + chronological split\n'
        f'Input filename: {Path(path).name}\n'
        f'Eligible homes (min 75 days of valid hourly data): {len(homes)}\n'
        f'Train homes: {train_homes}\n'
        f'Never-seen test homes: {heldout_homes}\n'
        f'Time cutoff: {cutoff}\n'
        f'Training observations: {len(train)}; test observations: {len(test)}\n'
        'Target: hourly kWh. Features use history from previous day or earlier.\n'
        'A strict group/time split is used: test homes not seen in training, test dates later.\n'
        'No weather features in this step. Weather is a separate optional later step.\n'
        'This is a London dataset: do not claim these are real Turkish homes.\n'
        'London and prior French single-home MAEs are not directly comparable due to different homes/splits.\n'
    )
    (output/'methodology_notes.txt').write_text(note,encoding='utf8')
    print(note)
    print(metrics.round(4).to_string(index=False))
    return metrics,per_home,test



## 4 — Modeli çalıştır ve sonuçları gör

In [ ]:
from IPython.display import display, Image
metrics, per_home, predictions = process(source_file, out_dir="London_Sonuclar")
print("✅ London deneyi tamamlandı")
display(metrics)
display(per_home)
display(Image(filename="London_Sonuclar/unseen_home_example.png"))


## 5 — Sonuçları indir ve buraya yükle

In [ ]:
from pathlib import Path
import shutil
from google.colab import files
zip_path = shutil.make_archive("London_Sonuclar", "zip", "London_Sonuclar")
print("✅ Sonuç dosyaları:")
for p in Path('London_Sonuclar').iterdir(): print(' -',p.name)
files.download(zip_path)
